<a href="https://colab.research.google.com/github/likithams09/deep-learning-with-python/blob/main/Program_3(MNIST).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 MNIST Dataset Augmentation Using A Deep Neural Network

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
Input,
Flatten,
Dense
)

 Load MNIST Dataset

In [4]:
(x_train, y_train), (x_test, y_test) = \
 tf.keras.datasets.mnist.load_data()
print("Original Training dataset:", x_train.shape)
print("Original Test dataset:", x_test.shape)

Original Training dataset: (60000, 28, 28)
Original Test dataset: (10000, 28, 28)


 Select 10,000 Original Training Images

In [5]:
N = 10000
x_original = x_train[:N]
y_original = y_train[:N]
print("\nSelected original images:", x_original.shape)
print("Selected original labels:", y_original.shape)


Selected original images: (10000, 28, 28)
Selected original labels: (10000,)


Create Rotated Images

In [6]:
# Convert images to floating point
x_float = x_original.astype("float32")
# Add channel dimension
x_float = x_float[..., np.newaxis]
# Rotate each image by 10 degrees
x_rotated = tf.keras.layers.RandomRotation(
factor=10 / 360,
fill_mode="constant"
)(x_float, training=True)
# Convert TensorFlow tensor to NumPy array
x_rotated = x_rotated.numpy()
# Remove channel dimension
x_rotated = np.squeeze(x_rotated, axis=-1)
# Convert back to uint8
x_rotated = np.clip(x_rotated, 0, 255).astype("uint8")
# Labels remain unchanged
y_rotated = y_original.copy()
print("\nRotated images:", x_rotated.shape)
print("Rotated labels:", y_rotated.shape)


Rotated images: (10000, 28, 28)
Rotated labels: (10000,)


Create Shifted Images

In [7]:
# Shift images horizontally and vertically
# using RandomTranslation
x_shifted = tf.keras.layers.RandomTranslation(
height_factor=0.10,
width_factor=0.10,
fill_mode="constant"
)(x_float, training=True)
# Convert TensorFlow tensor to NumPy
x_shifted = x_shifted.numpy()
# Remove channel dimension
x_shifted = np.squeeze(x_shifted, axis=-1)
# Keep pixel values between 0 and 255
x_shifted = np.clip(x_shifted, 0, 255).astype("uint8")
# Labels remain unchanged
y_shifted = y_original.copy()
print("Shifted images:", x_shifted.shape)
print("Shifted labels:", y_shifted.shape)

Shifted images: (10000, 28, 28)
Shifted labels: (10000,)


Combine Original + Rotated + Shifted Data

In [9]:
x_augmented = np.concatenate(
    [
        x_original,
        x_rotated,
        x_shifted
    ],
    axis=0
)
y_augmented = np.concatenate(
    [
        y_original,
        y_rotated,
        y_shifted
    ],
    axis=0
)

Display Dataset Size

In [10]:
print("Original images :", len(x_original))
print("Rotated images  :", len(x_rotated))
print("Shifted images  :", len(x_shifted))
print("Final images    :", len(x_augmented))
print("Final labels    :", len(y_augmented))

Original images : 10000
Rotated images  : 10000
Shifted images  : 10000
Final images    : 30000
Final labels    : 30000


Normalize the Augmented Data

In [11]:
x_augmented = x_augmented.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

Create Deep Neural Network

In [12]:
model = Sequential([
# INPUT LAYER
Input(shape=(28, 28)),
# Convert 28 x 28 into 784
Flatten(),
# HIDDEN LAYER 1
Dense(
128,
activation="relu"
),
# HIDDEN LAYER 2
Dense(
64,
activation="relu"
),
# OUTPUT LAYER
Dense(
10,
activation="softmax"
)
])

 Display Architecture

In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

Compile the Model


In [14]:
model.compile(
optimizer="sgd",
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)

Train the Model

In [15]:
history = model.fit(
x_augmented,
y_augmented,
epochs=5,
batch_size=128,
validation_split=0.1
)

Epoch 1/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4752 - loss: 1.8598 - val_accuracy: 0.4383 - val_loss: 1.7614
Epoch 2/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7059 - loss: 1.1268 - val_accuracy: 0.5043 - val_loss: 1.4975
Epoch 3/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7625 - loss: 0.8440 - val_accuracy: 0.5377 - val_loss: 1.3993
Epoch 4/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7904 - loss: 0.7279 - val_accuracy: 0.5647 - val_loss: 1.3399
Epoch 5/5
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8068 - loss: 0.6628 - val_accuracy: 0.5830 - val_loss: 1.2911


Test the Model

In [16]:
test_loss, test_accuracy = model.evaluate(
x_test,
y_test,
verbose=0
)

Display Final Result

In [17]:
print("Test Loss     :", test_loss)
print("Test Accuracy :", test_accuracy)

Test Loss     : 0.43084022402763367
Test Accuracy : 0.8906000256538391
